# Optional extension — Forced oscillations and resonance

## Zorlanmış salınımlar ve rezonans

**Role in the course:** Supports Week 13 (periodic motion). Driven damped oscillators, resonance curves and the quality factor.

This notebook is **not** a scheduled calendar week. Use it for deeper reading, extra examples and practice after the related weekly notebook. Its problems keep the identifier *Module 12 Pn* for the solution collection.

**TR:** Bu not takvimde ayrı bir hafta değildir; ilgili haftadan sonra ek okuma ve alıştırma için kullanılır.

## Contents / İçindekiler

1. [Before you start](#x12-before)
   · [Setup for the interactive graphs (run once)](#x12-setup)
2. [Concepts, demonstrations and worked examples](#x12-concepts) — 3 worked examples, 5 interactive graphs
3. [Problem set — predict, then check](#x12-problems) — 10 problems

[Course page / Ders sayfası](https://arifsolmaz.github.io/courses/fall/phy101/web/PHY101_Course_Dashboard.html)


<a id="x12-before"></a>

## 1. Before you start / Başlamadan önce

### Learning Objectives

By the end of this notebook, you will be able to:

1. **Write** the equation of motion for a driven (forced) damped harmonic oscillator
2. **Derive** and interpret the steady-state amplitude and phase response
3. **Explain** the resonance phenomenon and identify the resonance frequency
4. **Calculate** the quality factor $Q$ and relate it to the sharpness of the resonance peak
5. **Analyze** amplitude-frequency and phase-frequency response curves
6. **Design** a simple vibration damper by selecting optimal damping parameters
7. **Simulate** driven oscillator dynamics using `scipy.integrate.odeint`

### Joining this lesson / Derse buradan başlayanlar

**Quick recap.** A linear oscillator has $\omega_0=\sqrt{\frac{k}{m}}$. Viscous damping adds a force $-bv$ and a decay rate $\gamma=\frac{b}{2m}$; a periodic force adds $F_0\cos(\Omega t)$. After the transient has decayed, its displacement amplitude is $A=F_0/\sqrt{(k-m\Omega^2)^2+(b\Omega)^2}$. This formula assumes linear spring/damping and a specified force input. Road or floor displacement is a different input model. Convert frequency using $\Omega=2\pi f$ and RPM to Hz by dividing by 60.

**One system, shared engineering questions.** For $m=1\,\mathrm{kg}$, $k=100\,\mathrm{N/m}$, $b=2\,\mathrm{N}$·s/m and a 1 N drive at $\Omega=10\,\mathrm{rad/s}$, $k-m\Omega^2=0$, so $A=1/(2\times10)=0.050\,\mathrm{m}$. A mechanical/mechatronics engineer can predict that doubling damping halves this amplitude at the same drive frequency. A software/computer engineer reviewing a vibration display can check that 10 rad/s was not entered as 10 Hz, and compare measured amplitude with the 50 mm prediction. The task is to justify the model and interpret the result.

**TR:** Sürme frekansı, doğal frekans ve sönüm ayrı büyüklüklerdir. Önce girişin ne olduğunu ve birimleri belirle; sonra genlik grafiğini yorumla.

---

### Before we calculate: driving, response and resonance / Zorlanmış titreşime geçiş

**Core route:** identify the driver → compare its frequency with the natural frequency → calculate amplitude/phase → interpret damping. Solving differential equations or programming a sweep is optional. The RLC analogy and tuned mass damper are extensions after the mechanical model is understood.

Use $\Omega$ for the **driving angular frequency**, $\omega_0=\sqrt{\frac{k}{m}}$ for the **undamped natural frequency**, and $\gamma=\frac{b}{2m}$ for the **decay rate**. Earlier cells sometimes call the driving frequency $\omega_d$; in the previous notebook that symbol meant the *damped free* frequency. Read the label, not only the letter. Similarly $F_0/m$ is an acceleration amplitude, whereas a frequency in Hz is $f_0=\omega_0/(2\pi)$.

The force-form amplitude is easy to calculate in a small table:

<table width="100%">
<thead>
<tr>
<th align="left" width="64" scope="col">Step</th>
<th align="left" width="96" scope="col">Calculate</th>
<th align="left" width="64" scope="col">Units</th>
</tr>
</thead>
<tbody>
<tr>
<td>1</td>
<td>$u=k-m\Omega^2$</td>
<td>$\mathrm{N/m}$</td>
</tr>
<tr>
<td>2</td>
<td>$v=b\Omega$</td>
<td>$\mathrm{N/m}$</td>
</tr>
<tr>
<td>3</td>
<td>$D=\sqrt{u^2+v^2}$</td>
<td>$\mathrm{N/m}$</td>
</tr>
<tr>
<td>4</td>
<td>$A=F_0/D$</td>
<td>$\mathrm{m}$</td>
</tr>
</tbody>
</table>

Keep parentheses: $(k-m\Omega^2)^2$ means square the whole difference. A negative $u$ is allowed; amplitude is always nonnegative. For phase, choose $0^\circ$–$90^\circ$ below $\omega_0$, $90^\circ$ at $\omega_0$, and $90^\circ$–$180^\circ$ above it.

**Five-minute pause:** At very slow driving, inertia is small and $A\to F_0/k$. At very fast driving, $A\sim F_0/(m\Omega^2)$. Use these limits to explain your graph before using its sliders. **TR:** Rezonans, dış etkinin cismi hangi frekansta sürdüğüne bağlıdır. Büyük bir grafik gördüğünde önce eksenleri, birimleri ve hangi genliğin çizildiğini oku.

<a id="x12-setup"></a>

## Setup for the interactive graphs (run once) / Kurulum — bir kez çalıştır

Run the cells in this section once per session, then run any **Run the demonstration** cell below. Each demonstration shows a status line under its controls: **Updating…** while the graph is drawn, then the draw time and whether the sliders update live or on release. Nothing here needs to be edited.  
**TR:** Bu bölümdeki hücreleri oturum başına bir kez çalıştır; sonra istediğin gösterimi çalıştır. Kontrollerin altındaki durum satırı, grafiğin ne zaman güncellendiğini gösterir.

In [ ]:
#@title Run once — prepare the physics demonstrations
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from IPython.display import HTML, display, clear_output
import ipywidgets as widgets
from scipy.integrate import odeint

plt.rcParams.update({
    'figure.figsize': (10, 6),
    'font.size': 12,
    'axes.grid': True,
    'grid.alpha': 0.3
})

print("All libraries loaded successfully!")

In [ ]:
#@title Run once — prepare the demonstration controls
"""Shared demonstration interface, embedded in every PHY101 notebook.

Only standard ipywidgets, IPython and matplotlib are used, so the notebooks stay
self-contained in Colab and in a local Jupyter. The design goals are:

* A slider change must always produce a visible reaction. The status line under
  the controls says "Updating…" immediately and reports the draw time afterwards.
* The graph is replaced through the same Output-widget route that
  ``ipywidgets.interact`` uses (``clear_output(wait=True)`` followed by a fresh
  display), which is the most widely tested path in Colab and Jupyter. No
  output-capturing context is used: ipykernel 7 dispatches widget messages
  concurrently and IPython's capture object breaks that dispatch.
* Live updates while dragging are switched on when a graph draws quickly and
  switched off (update on release) when it draws slowly, so the kernel never
  falls behind a fast slider.
"""
import functools
import sys
import time
import traceback

import ipywidgets as widgets
from IPython import get_ipython
from IPython.display import HTML, clear_output, display

# Force the inline backend. Otherwise a local kernel may choose a desktop
# backend and block at plt.show(), which looks like a frozen notebook.
if get_ipython() is not None:
    get_ipython().run_line_magic("matplotlib", "inline")

_physics_panels = []
_physics_callback_errors = []

# Draw-time thresholds (seconds) for switching live dragging on and off.
PHYSICS_LIVE_ON = 0.12
PHYSICS_LIVE_OFF = 0.25


def physics_frames(frame_count, maximum=60):
    """Sample display frames, retaining both endpoints and all simulation data."""
    count = int(frame_count)
    shown = min(count, maximum)
    if shown <= 1:
        return list(range(shown))
    return [round(index * (count - 1) / (shown - 1)) for index in range(shown)]


def physics_interval(frame_count, interval_ms):
    """Preserve first-to-last playback duration when display frames are sampled."""
    shown = len(physics_frames(frame_count))
    return interval_ms if shown <= 1 else interval_ms * (int(frame_count) - 1) / (shown - 1)


PHYSICS_STYLE = """<style>
.phy101-panel { border: 1px solid #a9b9c9; border-radius: 8px; padding: 8px; background: #fff; }
.phy101-controls { padding: 0 0 2px; box-sizing: border-box; }
.phy101-status { font-size: 12px; color: #4a5a6a; padding: 0 2px 6px; min-height: 18px; }
.phy101-status.busy { color: #b45309; }
.phy101-plot-output img { max-width: 100%; height: auto; object-fit: contain; }
.phy101-plot-output .output_area { overflow: visible; }
.phy101-plot-output table { font-size: 13px; width: 100%; }
.phy101-plot-output .animation { max-width: 100%; }
.phy101-plot-output .animation img { max-width: 100%; height: auto; object-fit: contain; }
.phy101-animation-panel { max-width: 100%; }
.phy101-animation-panel .animation { display: flex; flex-direction: column; }
.phy101-animation-panel .animation img { order: 2; max-width: 100%; height: auto; }
.phy101-animation-panel .anim-controls { order: 1; background: white; color: #172433; padding: 4px; }
@media (max-width: 650px) {
  .phy101-controls, .phy101-plot-output { width: 100% !important; }
}
</style>"""
display(HTML(PHYSICS_STYLE))


def physics_animation_html(animation):
    """A self-contained animation pane with playback controls kept in view."""
    return HTML(PHYSICS_STYLE + '<div class="phy101-animation-panel">' +
                animation.to_jshtml(default_mode="once") + "</div>")


def _physics_controls(items):
    """Lay the controls out as a wrapping toolbar with full-length labels."""
    flat = []
    for item in items:
        if isinstance(item, (widgets.HBox, widgets.VBox)):
            flat.extend(item.children)
        else:
            flat.append(item)
    for control in flat:
        if hasattr(control, "style") and "description_width" in control.style.traits():
            control.style.description_width = "initial"
        control.layout.width = "310px"
        control.layout.flex = "0 1 310px"
        control.layout.max_width = "100%"
        control.layout.min_width = "0"
        control.layout.margin = "2px 6px 2px 0"
        if isinstance(control, widgets.Button):
            control.layout.width = "auto"
            control.layout.flex = "0 0 auto"
    # Repeat the style inside the widget tree: Colab isolates output frames.
    style = widgets.HTML(value=PHYSICS_STYLE, layout=widgets.Layout(display="none"))
    box = widgets.Box([style] + flat, layout=widgets.Layout(
        display="flex", flex_flow="row wrap", align_items="center",
        width="100%", min_width="0", max_width="100%"))
    box.add_class("phy101-controls")
    return box


def _physics_output(output):
    output.layout = widgets.Layout(
        width="100%", min_width="0", max_width="100%",
        height="auto", overflow="visible", margin="0")
    output.add_class("phy101-plot-output")
    return output


def physics_panel(controls, output):
    """Button-driven demos: a compact control toolbar directly above the result."""
    panel = widgets.Box([_physics_controls(controls), _physics_output(output)],
        layout=widgets.Layout(display="flex", flex_flow="column",
                              align_items="stretch", width="100%"))
    panel.add_class("phy101-panel")
    return panel


def physics_show_figure(figure):
    """Display one inline figure and close its pyplot registration afterwards."""
    import matplotlib.pyplot as plt
    display(figure)
    plt.close(figure)


def physics_vector_axes(axes, points):
    """Equal x/y scales and limits covering all arrow endpoints, including sums."""
    import numpy as np
    coordinates = np.asarray(points, dtype=float).reshape(-1, 2)
    span = max(1.0, float(np.max(np.abs(coordinates)))) * 1.22
    axes.set(xlim=(-span, span), ylim=(-span, span), xlabel="x component", ylabel="y component")
    axes.set_aspect("equal", adjustable="box")
    axes.axhline(0, color="#718096", linewidth=0.7)
    axes.axvline(0, color="#718096", linewidth=0.7)
    axes.grid(alpha=0.2)


class PhysicsPanel:
    """Controls, a status line and one Output widget that shows the latest result."""

    def __init__(self, function, controls):
        self.f = function
        self.controls = controls
        self.out = _physics_output(widgets.Output())
        self.status = widgets.HTML(value="")
        self.status.add_class("phy101-status")
        self.seconds = None
        self.live = True
        self.updates = 0
        self.last_outputs = 0
        self.figures = 0
        self.error = None
        visible = []
        for control in controls.values():
            if isinstance(control, widgets.fixed):
                continue
            visible.append(control)
            if hasattr(control, "continuous_update"):
                control.continuous_update = True
            control.observe(self._changed, names="value")
        self.widget = widgets.VBox([_physics_controls(visible), self.status, self.out],
                                   layout=widgets.Layout(width="100%"))
        self.widget.add_class("phy101-panel")
        self.children = self.widget.children
        _physics_panels.append(self)
        self.render()

    # Compatibility with the earlier validation code.
    @property
    def layout(self):
        return self.widget.layout

    def _changed(self, change):
        self.render()

    def _set_live(self, live):
        if live == self.live:
            return
        self.live = live
        for control in self.controls.values():
            if hasattr(control, "continuous_update"):
                control.continuous_update = live

    def render(self):
        self.status.value = "⏳ Updating… / Güncelleniyor…"
        self.status.add_class("busy")
        started = time.perf_counter()
        kwargs = {name: control.value for name, control in self.controls.items()}
        self.error = None
        self.figures = 0
        # Count everything the demonstration shows (figures, HTML, animations, text)
        # by wrapping the display publisher and stdout for this draw only.
        shell = get_ipython()
        publisher = getattr(shell, "display_pub", None) if shell is not None else None
        if publisher is not None:
            original_publish = publisher.publish

            def counting_publish(*args, **kwargs):
                self.figures += 1
                return original_publish(*args, **kwargs)
            publisher.publish = counting_publish
        stdout = sys.stdout
        original_write = stdout.write

        def counting_write(text):
            if text.strip():
                self.figures += 1
            return original_write(text)
        stdout.write = counting_write
        try:
            # The previous result stays visible until the new one arrives.
            with self.out:
                clear_output(wait=True)
                try:
                    result = self.f(**kwargs)
                    from ipywidgets.widgets.interaction import show_inline_matplotlib_plots
                    show_inline_matplotlib_plots()
                    if result is not None:
                        display(result)
                except Exception:
                    self.error = traceback.format_exc()
                    _physics_callback_errors.append((getattr(self.f, "__name__", "callback"), self.error))
                    print(self.error)
        finally:
            if publisher is not None and publisher.__dict__.get("publish") is counting_publish:
                del publisher.publish
            if stdout.__dict__.get("write") is counting_write:
                del stdout.write
        self.last_outputs = self.figures
        self.seconds = time.perf_counter() - started
        self.updates += 1
        if self.seconds > PHYSICS_LIVE_OFF:
            self._set_live(False)
        elif self.seconds < PHYSICS_LIVE_ON:
            self._set_live(True)
        self.status.remove_class("busy")
        mode = ("updates while you drag / sürüklerken güncellenir" if self.live
                else "updates when you release the slider / kaydırıcıyı bırakınca güncellenir")
        self.status.value = (f"✓ Drawn in {self.seconds:.2f} s · {mode}" if self.error is None
                             else "⚠ The demonstration reported an error; see the message below.")


def physics_interactive(function, **controls):
    """Build a panel like ipywidgets.interactive, returning the panel object."""
    @functools.wraps(function)
    def checked(*args, **kwargs):
        return function(*args, **kwargs)

    return PhysicsPanel(checked, controls)


def physics_interact(function=None, **controls):
    """Support both @physics_interact(...) and physics_interact(function, ...)."""
    if function is None:
        return lambda function: physics_interact(function, **controls)
    panel = physics_interactive(function, **controls)
    function.widget = panel.widget
    function.panel = panel
    display(panel.widget)
    return function


<a id="x12-concepts"></a>

## 2. Concepts, demonstrations and worked examples / Konular, gösterimler ve çözümlü örnekler

### The Driven Damped Harmonic Oscillator

Last week we studied free oscillations (with and without damping). Now we add a **periodic driving force**:

$$m\ddot{x} + b\dot{x} + kx = F_0 \cos(\omega_d t)$$

Dividing by $m$ and defining $\gamma = \frac{b}{2m}$, $\omega_0 = \sqrt{\frac{k}{m}}$, $f_0 = F_0/m$:

$$\ddot{x} + 2\gamma\dot{x} + \omega_0^2 x = f_0 \cos(\omega_d t)$$

#### Analogy

Imagine pushing a child on a swing. You apply a periodic push (the driving force). If you push at the right moment (in sync with the natural swing frequency), the amplitude builds up dramatically. Push at the wrong frequency, and the swing barely responds. This is **resonance**.

#### Steady-State Solution

After transients die out, the system settles into a steady-state oscillation at the **driving frequency** $\omega_d$:

$$x(t) = A(\omega_d)\cos(\omega_d t - \delta)$$

where the **steady-state amplitude** is:

$$A(\omega_d) = \frac{f_0}{\sqrt{(\omega_0^2 - \omega_d^2)^2 + (2\gamma\omega_d)^2}}$$

and the **phase lag** is:

$$\delta = \operatorname{atan2}\!\left(2\gamma\omega_d,\,\omega_0^2 - \omega_d^2\right),\quad 0\le\delta\le\pi$$

#### Key Features

<table width="100%">
<thead>
<tr>
<th align="left" width="216" scope="col">Feature</th>
<th align="left" width="800" scope="col">Description</th>
</tr>
</thead>
<tbody>
<tr>
<td><strong>Resonance frequency</strong></td>
<td>$\omega_r = \sqrt{\omega_0^2 - 2\gamma^2}$ (displacement peaks here only when $\gamma<\omega_0/\sqrt2$)</td>
</tr>
<tr>
<td><strong>At $\omega_d=\omega_0$</strong></td>
<td>Phase lag $\delta=90^\circ$; power is maximal. The displacement peak lies slightly lower for nonzero damping.</td>
</tr>
<tr>
<td><strong>Low frequency</strong> ($\omega_d \ll \omega_0$)</td>
<td>$A \approx f_0/\omega_0^2 = F_0/k$, $\delta \approx 0^\circ$</td>
</tr>
<tr>
<td><strong>High frequency</strong> ($\omega_d \gg \omega_0$)</td>
<td>$A \to 0$, $\delta \to 180^\circ$</td>
</tr>
</tbody>
</table>

#### Worked force balance: three denominator steps / Paydayı parçalara ayır

Use $m=1.0\,\mathrm{kg}$, $k=100\,\mathrm{N/m}$, $b=2.0\,\mathrm{N\,s/m}$ and a force $F(t)=(1.0\,\mathrm N)\cos[(10\,\mathrm{rad/s})t]$. Here the driving frequency is $\Omega=10\,\mathrm{rad/s}$, not $10\,\mathrm{Hz}$.

$$u=k-m\Omega^2=100-1(10)^2=0\,\mathrm{N/m},\qquad
v=b\Omega=2(10)=20\,\mathrm{N/m}.$$
$$D=\sqrt{u^2+v^2}=20\,\mathrm{N/m},\qquad
A=\frac{F_0}{D}=\frac{1.0\,\mathrm N}{20\,\mathrm{N/m}}
=0.050\,\mathrm m=50\,\mathrm{mm}.$$

The dynamic stiffness and inertia terms cancel at this driving frequency; damping limits the steady amplitude. The phase lag is $\delta=\pi/2=90^\circ$. This is the natural-frequency/power-resonance condition; a damped displacement peak is slightly lower in frequency.

**Think–Pair–Explain:** Double $b$ while keeping the same drive. **Worked response:** $D$ becomes $40\,\mathrm{N/m}$, so $A=1/40=0.025\,\mathrm m$. More energy is dissipated for a given velocity, so the steady response is smaller.

**Türkçe:** Önce sürme frekansını kuvvet ifadesinden oku. Paydadaki farkın tamamını karesini almadan önce hesapla. Burada fark sıfır olduğu için sönüm terimi kalır. $\mathrm N/(\mathrm{N/m})=\mathrm m$ birim kontrolü, bulduğumuz büyüklüğün genlik olduğunu gösterir.

### Interactive: Driven Oscillator Animation with Resonance Buildup

Watch how a driven oscillator builds up amplitude over time. The transient (natural frequency) eventually dies out, leaving only the steady-state response at the driving frequency.

In [ ]:
#@title Run the animation — observe the physical motion
def driven_oscillator_animation(omega_d=None, gamma=0.15, omega0=2*np.pi, F0_over_m=5.0):
    """Animated driven oscillator showing resonance buildup."""
    if omega_d is None:
        omega_d = omega0  # Drive at resonance by default

    def driven_ode(state, t):
        x, v = state
        driving = F0_over_m * np.cos(omega_d * t)
        return [v, driving - 2 * gamma * v - omega0**2 * x]

    t_end = 30.0
    t_arr = np.linspace(0, t_end, 1500)
    sol = odeint(driven_ode, [0, 0], t_arr)
    x_arr = sol[:, 0]
    v_arr = sol[:, 1]

    # Steady-state amplitude
    A_ss = F0_over_m / np.sqrt((omega0**2 - omega_d**2)**2 + (2 * gamma * omega_d)**2)
    driving_arr = F0_over_m * np.cos(omega_d * t_arr) / omega0**2  # normalized

    fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(12, 10), layout="constrained")
    plt.close(fig)

    # --- Top: Driving force ---
    ax1.set_xlim(0, t_end)
    force_max = F0_over_m / omega0**2 * 1.3
    ax1.set_ylim(-force_max, force_max)
    ax1.set_ylabel('Force / k (m)', fontsize=11)
    ax1.set_title('Driving Force', fontweight='bold')
    drive_line, = ax1.plot([], [], 'orange', lw=2)

    # --- Middle: Response ---
    ax2.set_xlim(0, t_end)
    x_max = max(np.abs(x_arr).max() * 1.2, A_ss * 1.3)
    ax2.set_ylim(-x_max, x_max)
    ax2.set_ylabel('x (m)', fontsize=11)
    ax2.set_title(f'Response ($\\omega_d$={omega_d:.2f}, $\\omega_0$={omega0:.2f}, A_ss={A_ss:.2f} m)', fontweight='bold')
    ax2.axhline(y=A_ss, color='green', ls='--', lw=1.5, alpha=0.5, label=f'Steady-state A = {A_ss:.2f}')
    ax2.axhline(y=-A_ss, color='green', ls='--', lw=1.5, alpha=0.5)
    response_line, = ax2.plot([], [], 'b-', lw=2, label='x(t)')
    ax2.legend(loc='upper center', bbox_to_anchor=(0.5, -0.20), ncol=2, fontsize=9)

    # --- Bottom: Spring-mass visual ---
    visual_span = max(1.0, x_max * 1.5 + 0.5)
    ax3.set_xlim(-visual_span, visual_span)
    ax3.set_ylim(-0.8, 0.8)
    ax3.set_aspect('equal')
    ax3.set_title('Mass on Spring (driven)', fontweight='bold')
    ax3.set_xlabel('Position x (m)')
    ax3.set_yticks([])
    ax3.axvline(0, color='gray', ls='--', alpha=0.4)

    wall = plt.Rectangle((-visual_span, -0.5), 0.1, 1.0, color='gray')
    ax3.add_patch(wall)
    spring_line, = ax3.plot([], [], 'b-', lw=2)
    mass_patch = plt.Rectangle((0, -0.25), 0.5, 0.5, fc='royalblue', ec='navy', lw=2)
    ax3.add_patch(mass_patch)
    force_arrow, = ax3.plot([], [], 'r-', lw=3)
    info_text = ax3.text(0.98, 0.95, '', transform=ax3.transAxes, va='top', fontsize=10, ha='right',
                          bbox=dict(boxstyle='round', fc='lightyellow'))

    def make_spring(x0, x1, n_coils=12, width=0.15):
        L = x1 - x0
        s = np.linspace(0, 1, n_coils * 20 + 1)
        sx = x0 + s * L
        sy = width * np.sin(2 * np.pi * n_coils * s)
        sy[0] = 0; sy[-1] = 0
        return sx, sy


    step = 3  # frame stepping for speed

    def animate(frame):
        i = frame * step
        if i >= len(t_arr):
            i = len(t_arr) - 1

        drive_line.set_data(t_arr[:i+1], driving_arr[:i+1])
        response_line.set_data(t_arr[:i+1], x_arr[:i+1])

        x = x_arr[i]
        sx, sy = make_spring(-visual_span + 0.1, x - 0.25)
        spring_line.set_data(sx, sy)
        mass_patch.set_xy((x - 0.25, -0.25))

        # Force arrow
        f_now = F0_over_m * np.cos(omega_d * t_arr[i])
        arrow_len = f_now / omega0**2 * 0.5
        force_arrow.set_data([x + 0.25, x + 0.25 + arrow_len], [0, 0])

        info_text.set_text(f't = {t_arr[i]:.1f} s\nx = {x:.3f} m')

        return drive_line, response_line, spring_line, mass_patch, force_arrow, info_text

    n_frames = len(t_arr) // step
    ani = animation.FuncAnimation(fig, animate, frames=physics_frames(n_frames),
                                  interval=physics_interval(n_frames, 25), blit=False)
    return physics_animation_html(ani)

print("Driving AT resonance (omega_d = omega_0):")
display(driven_oscillator_animation(omega_d=2*np.pi, gamma=0.15, omega0=2*np.pi))

In [ ]:
#@title Optional numerical check — the algebra is explained above
print("Driving AWAY from resonance (omega_d = 1.5 * omega_0):")
display(driven_oscillator_animation(omega_d=3*np.pi, gamma=0.15, omega0=2*np.pi))

### The Resonance Curve

The amplitude response function:

$$A(\omega_d) = \frac{f_0}{\sqrt{(\omega_0^2 - \omega_d^2)^2 + (2\gamma\omega_d)^2}}$$

This curve has a peak near $\omega_0$. The **width** and **height** of this peak are controlled by the damping $\gamma$:

- **Light damping** ($\gamma \ll \omega_0$): tall, narrow peak
- **Heavy damping** ($\gamma \sim \omega_0$): short, broad peak

The peak occurs at the **resonance frequency**:

$$\omega_r = \sqrt{\omega_0^2 - 2\gamma^2}$$

This positive-frequency displacement peak exists only if $\gamma<\omega_0/\sqrt2$. Otherwise the largest displacement response occurs at zero frequency. For light damping, $\omega_r\approx\omega_0$.

**Concrete comparison:** with $\omega_0=10\,\mathrm{rad/s}$ and $\gamma=1\,\mathrm{s^{-1}}$,
$$\omega_r=\sqrt{10^2-2(1)^2}=\sqrt{98}=9.90\,\mathrm{rad/s}.$$
With $\gamma=8\,\mathrm{s^{-1}}$, the expression under the square root is $100-128=-28\,\mathrm{s^{-2}}$. This means there is **no positive-frequency displacement peak**, not an imaginary driving frequency to plot.

**Türkçe:** Formülün karekök içi negatif çıktığında fiziksel koşulu kontrol et. Her sönüm değerinde bir rezonans tepesi bulunmaz. Doğal frekans, sönümlü serbest frekans ve genlik tepesinin frekansı farklı tanımlardır.


#### ⏱️ Checkpoint 1 of 3 — Think · Pair · Explain

**Small class activity / Kısa sınıf etkinliği.** A $1\,\mathrm{kg}$ mass on a $100\,\mathrm{N/m}$ spring is driven at $10\,\mathrm{rad/s}$ with $F_0=1\,\mathrm N$. Damping changes from $b=2$ to $4\,\mathrm{N\,s/m}$.

**Think — 1 minute:** Predict whether the steady displacement grows or shrinks.

**Pair — 2 minutes:** Compare your sign, units and first equation. Explain a disagreement before checking the model response.

**Explain — 2 minutes:** Write the physical reason for your answer; the calculation alone is not the explanation.

**Worked model response / Adım adım örnek yanıt**

At this frequency, $k-m\Omega^2=0$, so
$$A=\frac{F_0}{b\Omega},\qquad A_1=\frac1{2(10)}=0.050\,\mathrm m,
\quad A_2=\frac1{4(10)}=0.025\,\mathrm m.$$
Doubling damping halves the amplitude at the same drive frequency. This calculation uses a force input and the steady state after transients decay.

**Türkçe:** Önce paydadaki yay ve atalet terimlerinin birbirini götürdüğünü göster. Sonra aynı frekansta sönümün iki katına çıkmasının genliği yarıya indirdiğini açıkla. Sonuç yol yüksekliğiyle sürülen farklı bir düzeneğe doğrudan taşınmaz.

**Your explanation / Açıklaman:** write or discuss your reasoning in words, with an equation or sketch where useful.

### Interactive: Amplitude vs Frequency Response Curve

Adjust the damping coefficient and observe how the resonance peak changes.

In [ ]:
#@title Run the demonstration — predict, adjust, observe
def interactive_resonance_curve(gamma, omega0, f0):
    """Amplitude-frequency response with adjustable damping."""
    omega_d = np.unique(np.r_[np.linspace(0, 3 * omega0, 1000),
                               np.sqrt(max(0, omega0**2 - 2 * gamma**2))])

    A_response = f0 / np.sqrt((omega0**2 - omega_d**2)**2 + (2 * gamma * omega_d)**2)

    # Phase
    delta = np.arctan2(2 * gamma * omega_d, omega0**2 - omega_d**2)

    # Resonance frequency
    disc = omega0**2 - 2 * gamma**2
    if disc > 0:
        omega_r = np.sqrt(disc)
        A_max = f0 / (2 * gamma * np.sqrt(omega0**2 - gamma**2))
    else:
        omega_r = 0
        A_max = f0 / omega0**2

    # Q factor
    Q = omega0 / (2 * gamma) if gamma > 0 else float('inf')

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), layout="constrained")

    # --- Amplitude response ---
    ax1.plot(omega_d / omega0, A_response, 'b-', lw=2.5)
    ax1.axvline(x=1.0, color='gray', ls='--', alpha=0.5, label=f'$\\omega_0$ = {omega0:.2f}')
    if disc > 0:
        ax1.axvline(x=omega_r / omega0, color='red', ls=':', lw=2,
                    label=f'$\\omega_r$ = {omega_r:.2f}')
        ax1.plot(omega_r / omega0, A_max, 'r*', ms=15, label=f'$A_{{max}}$ = {A_max:.2f}')

    # Displacement-amplitude reference; not the exact half-power bandwidth
    A_half = A_max / np.sqrt(2) if disc > 0 else 0
    if disc > 0:
        ax1.axhline(y=A_half, color='green', ls=':', alpha=0.5, label='Peak amplitude / sqrt(2)')

    ax1.set_xlabel('$\\omega_d / \\omega_0$', fontsize=13)
    ax1.set_ylabel('Amplitude A (m)', fontsize=13)
    ax1.set_title(f'Amplitude Response\n$\\gamma$={gamma:.2f}, Q={Q:.1f}', fontweight='bold', fontsize=13)
    ax1.legend(loc='upper center', bbox_to_anchor=(0.5, -0.20), ncol=2, fontsize=9)
    ax1.set_xlim(0, 3)

    # --- Phase response ---
    ax2.plot(omega_d / omega0, np.degrees(delta), 'r-', lw=2.5)
    ax2.axvline(x=1.0, color='gray', ls='--', alpha=0.5)
    ax2.axhline(y=90, color='green', ls=':', alpha=0.5, label='$\\delta = 90°$')
    ax2.set_xlabel('$\\omega_d / \\omega_0$', fontsize=13)
    ax2.set_ylabel('Phase lag $\\delta$ (degrees)', fontsize=13)
    ax2.set_title('Phase Response', fontweight='bold', fontsize=13)
    ax2.set_ylim(-5, 185)
    ax2.set_xlim(0, 3)
    ax2.legend(loc='upper center', bbox_to_anchor=(0.5, -0.20), ncol=2, fontsize=9)

    plt.show()

physics_interact(interactive_resonance_curve,
    gamma=widgets.FloatSlider(value=0.3, min=0.05, max=5.0, step=0.05,
                              description='Damping $\\gamma$ (s$^{-1}$):',
                              style={'description_width': 'initial'}),
    omega0=widgets.FloatSlider(value=5.0, min=1.0, max=15.0, step=0.5,
                               description='$\\omega_0$ (rad/s):',
                               style={'description_width': 'initial'}),
    f0=widgets.FloatSlider(value=10.0, min=1.0, max=50.0, step=1.0,
                           description='$f_0 = F_0/m$ (m/s$^2$):',
                           style={'description_width': 'initial'})
);

#### Multiple Damping Values on One Plot

In [ ]:
#@title Optional numerical check — the algebra is explained above
omega0 = 5.0
f0 = 10.0
gamma_values = [0.1, 0.3, 0.7, 1.5, 3.0]
colors = plt.cm.coolwarm(np.linspace(0, 1, len(gamma_values)))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), layout="constrained")

omega_d = np.linspace(0.01, 3 * omega0, 1000)

for gam, col in zip(gamma_values, colors):
    A_resp = f0 / np.sqrt((omega0**2 - omega_d**2)**2 + (2 * gam * omega_d)**2)
    delta = np.arctan2(2 * gam * omega_d, omega0**2 - omega_d**2)
    Q = omega0 / (2 * gam)

    ax1.plot(omega_d / omega0, A_resp, color=col, lw=2,
             label=f'$\\gamma$={gam:.1f} (Q={Q:.1f})')
    ax2.plot(omega_d / omega0, np.degrees(delta), color=col, lw=2,
             label=f'$\\gamma$={gam:.1f}')

ax1.axvline(x=1.0, color='gray', ls='--', alpha=0.4)
ax1.set_xlabel('$\\omega_d / \\omega_0$', fontsize=13)
ax1.set_ylabel('Amplitude A (m)', fontsize=13)
ax1.set_title('Amplitude-Frequency Response', fontweight='bold', fontsize=14)
ax1.legend(loc='upper center', bbox_to_anchor=(0.5, -0.20), ncol=2, fontsize=9)
ax1.set_xlim(0, 3)

ax2.axvline(x=1.0, color='gray', ls='--', alpha=0.4)
ax2.axhline(y=90, color='gray', ls=':', alpha=0.4)
ax2.set_xlabel('$\\omega_d / \\omega_0$', fontsize=13)
ax2.set_ylabel('Phase lag $\\delta$ (degrees)', fontsize=13)
ax2.set_title('Phase-Frequency Response', fontweight='bold', fontsize=14)
ax2.legend(loc='upper center', bbox_to_anchor=(0.5, -0.20), ncol=2, fontsize=9)
ax2.set_xlim(0, 3)
ax2.set_ylim(-5, 185)

plt.show()

### Phase Diagram: Driving Frequency vs Response Phase

The phase relationship between the driving force and the response is crucial for understanding energy transfer:

- $\delta \approx 0^\circ$: Force and displacement are in phase (stiffness-dominated)
- $\delta = 90^\circ$: Force leads displacement by 90° (resonance, maximum power input)
- $\delta \approx 180^\circ$: Force and displacement are out of phase (inertia-dominated)

### Interactive: Phase Diagram

Visualize how the response (blue) relates to the driving force (orange) at different frequencies.

In [ ]:
#@title Run the demonstration — predict, adjust, observe
def interactive_phase_diagram(omega_d_ratio, gamma):
    """Show driving force vs response with phase relationship."""
    omega0 = 5.0
    f0 = 10.0
    omega_d = omega_d_ratio * omega0

    A_ss = f0 / np.sqrt((omega0**2 - omega_d**2)**2 + (2 * gamma * omega_d)**2)
    delta = np.arctan2(2 * gamma * omega_d, omega0**2 - omega_d**2)

    t = np.linspace(0, 4 * 2 * np.pi / omega_d, 500)
    force = np.cos(omega_d * t)
    response = A_ss * np.cos(omega_d * t - delta)

    # Also solve ODE for transient
    def driven_ode(state, t):
        x, v = state
        return [v, f0 * np.cos(omega_d * t) - 2 * gamma * v - omega0**2 * x]

    t_long = np.linspace(0, 40, 3000)
    sol = odeint(driven_ode, [0, 0], t_long)

    fig, axes = plt.subplots(2, 2, figsize=(14, 9), layout="constrained")

    # --- Steady state comparison ---
    ax = axes[0, 0]
    ax.plot(t, force, 'orange', lw=2, label='Driving force (normalized)')
    ax.plot(t, response / A_ss, 'b-', lw=2, label='Response (normalized)')
    ax.set_xlabel('Time (s)')
    ax.set_title(f'Steady State: $\\delta$ = {np.degrees(delta):.1f}°', fontweight='bold')
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.20), ncol=2, fontsize=9)

    # Phase arrow
    ax.annotate('', xy=(delta / omega_d, 1.2), xytext=(0, 1.2),
                arrowprops=dict(arrowstyle='<->', color='red', lw=2))
    ax.set_ylim(-1.15, 1.55)
    ax.text(delta / (2 * omega_d), 1.32, f'$\\delta$={np.degrees(delta):.0f}°',
            color='red', fontsize=11, ha='center')

    # --- Full transient ---
    ax = axes[0, 1]
    ax.plot(t_long, sol[:, 0], 'b-', lw=1.5)
    ax.axhline(y=A_ss, color='green', ls='--', alpha=0.5, label=f'$A_{{ss}}$={A_ss:.2f}')
    ax.axhline(y=-A_ss, color='green', ls='--', alpha=0.5)
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('x (m)')
    ax.set_title('Full Response (transient + steady)', fontweight='bold')
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.20), ncol=2, fontsize=9)

    # --- Phasor diagram ---
    ax = axes[1, 0]
    ax.set_xlim(-1.5, 1.5)
    ax.set_ylim(-1.5, 1.5)
    ax.set_aspect('equal')
    circle = plt.Circle((0, 0), 1, fill=False, color='gray', ls='--', alpha=0.3)
    ax.add_patch(circle)

    # Force phasor
    ax.arrow(0, 0, 0.9 * np.cos(0), 0.9 * np.sin(0), head_width=0.08,
             fc='orange', ec='orange', lw=2, label='Driving force')

    # Response phasor (lags by delta)
    ax.arrow(0, 0, 0.9 * np.cos(-delta), 0.9 * np.sin(-delta), head_width=0.08,
             fc='blue', ec='blue', lw=2, label='Displacement')

    # Arc for phase angle
    arc_theta = np.linspace(-delta, 0, 50)
    ax.plot(0.5 * np.cos(arc_theta), 0.5 * np.sin(arc_theta), 'r-', lw=2)
    ax.text(0.03, 0.95,
            f'$\\delta$={np.degrees(delta):.0f}°', color='red', fontsize=12, transform=ax.transAxes, va='top')
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.20), ncol=2, fontsize=9)

    ax.set_title('Phasor Diagram', fontweight='bold')

    # --- Power ---
    ax = axes[1, 1]
    power = f0 * np.cos(omega_d * t) * (-A_ss * omega_d * np.sin(omega_d * t - delta))
    ax.plot(t, power, 'm-', lw=2)
    ax.fill_between(t, 0, power, where=(power > 0), alpha=0.3, color='green', label='Energy in')
    ax.fill_between(t, 0, power, where=(power < 0), alpha=0.3, color='red', label='Energy out')
    P_avg = 0.5 * f0 * A_ss * omega_d * np.sin(delta)
    ax.axhline(y=P_avg, color='black', ls='--', lw=2, label=f'$\\langle P \\rangle$ = {P_avg:.2f}')
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Power (W/kg)')
    ax.set_title(f'Instantaneous Power', fontweight='bold')
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.20), ncol=2, fontsize=9)

    fig.suptitle(f'$\\omega_d/\\omega_0$ = {omega_d_ratio:.2f}, $\\gamma$ = {gamma:.2f}, A = {A_ss:.3f} m',
                 fontsize=14, fontweight='bold')
    plt.show()

physics_interact(interactive_phase_diagram,
    omega_d_ratio=widgets.FloatSlider(value=1.0, min=0.1, max=3.0, step=0.05,
                                      description='$\\omega_d/\\omega_0$:',
                                      style={'description_width': 'initial'}),
    gamma=widgets.FloatSlider(value=0.3, min=0.05, max=3.0, step=0.05,
                              description='$\\gamma$ (s$^{-1}$):',
                              style={'description_width': 'initial'})
);

### The Quality Factor (Q)

The **quality factor** $Q$ measures how "sharp" the resonance is. It is defined as:

$$Q = \frac{\omega_0}{2\gamma} = \frac{\omega_0}{\Delta\omega}$$

where $\Delta\omega$ is the **full width at half-maximum power** (FWHM) of the resonance curve.

<table width="100%">
<thead>
<tr>
<th align="left" width="136" scope="col">$Q$ value</th>
<th align="left" width="472" scope="col">Interpretation</th>
<th align="left" width="264" scope="col">Example</th>
</tr>
</thead>
<tbody>
<tr>
<td>$Q < 1/2$</td>
<td>Overdamped free motion</td>
<td>Strong viscous damping</td>
</tr>
<tr>
<td>$Q = 1/2$</td>
<td>Critical damping</td>
<td>Boundary of oscillatory return</td>
</tr>
<tr>
<td>$1/2 < Q \le 1/\sqrt2$</td>
<td>Underdamped, but no positive-frequency displacement peak</td>
<td>Broad response</td>
</tr>
<tr>
<td>$Q \sim 1-10$</td>
<td>Moderate damping</td>
<td>Car suspension</td>
</tr>
<tr>
<td>$Q \sim 100$</td>
<td>Sharp resonance</td>
<td>Acoustic guitar</td>
</tr>
<tr>
<td>$Q \sim 10^4$</td>
<td>Very sharp resonance</td>
<td>Quartz crystal oscillator</td>
</tr>
</tbody>
</table>

Physical meaning: $Q$ is approximately $2\pi$ times the number of oscillation cycles before the energy drops to $1/e$ of its initial value.

$$Q \approx 2\pi \times \frac{\text{Energy stored}}{\text{Energy lost per cycle}}$$

For $Q>1/\sqrt2$, a positive-frequency displacement peak exists. The $Q=\omega_0/\Delta\omega$ bandwidth relation here refers to the **mean dissipated-power** curve, not the half-height width of the displacement curve. For weak damping the two resonance frequencies are close. **TR:** Az sönümlü olmak ile genlik grafiğinde tepe olması aynı koşul değildir.

#### Read the bandwidth from the correct vertical axis / Bant genişliğinin ekseni

For the mean-power response, let $\omega_0=12\,\mathrm{rad/s}$ and $\gamma=1.5\,\mathrm{s^{-1}}$.
$$Q=\frac{12}{2(1.5)}=4,\qquad \Delta\omega=2\gamma=3.0\,\mathrm{rad/s}.$$
The half-power angular frequencies are
$$\omega_\pm=\sqrt{\omega_0^2+\gamma^2}\pm\gamma
=\sqrt{146}\pm1.5,
\quad\omega_-=10.6,\quad\omega_+=13.6\,\mathrm{rad/s}.$$
Subtracting the two limits gives $13.6-10.6=3.00\,\mathrm{rad/s}$. Half power is not half displacement amplitude. The following demonstration now labels its left vertical axis as **normalized mean power** for this bandwidth comparison.

**Türkçe:** Bant genişliğini hangi eğriden ölçtüğünü söyle. Gücün yarıya indiği iki frekansın farkını kullanıyoruz; yer değiştirme genliğinin yarısını almak aynı işlem değildir. $Q$ birimsizdir, çünkü pay ve payda aynı frekans birimine sahiptir.

**Why the two plots differ.** The bandwidth is measured on normalized **mean power**, not simply on displacement amplitude. For viscous damping with a fixed sinusoidal force,

$$\Omega_{\pm}=\sqrt{\omega_0^2+\gamma^2}\pm\gamma,\qquad \Delta\Omega=\Omega_+-\Omega_-=2\gamma.$$

The energy plot computes $E=\tfrac12mv^2+\tfrac12kx^2$ from a mass released from rest. The smooth $e^{-2\gamma t}$ line is a weak-damping guide, not its exact instantaneous energy at every point. Türkçe: $Q=0.5$ kritik sönümdür; bu durumda “kaç salınım” demek uygun değildir. Yatay eksendeki $t/T_0$ yalnız bir referans zaman ölçüsüdür.


### Interactive: Q Factor Visualizer

See how $Q$ controls the height and width of the resonance peak. The bandwidth (FWHM) is highlighted.

In [ ]:
#@title Run the demonstration — predict, adjust, observe
def interactive_Q_factor(Q):
    """Visualize the Q factor and bandwidth."""
    omega0 = 10.0
    f0 = 10.0
    gamma = omega0 / (2 * Q)

    # Mean dissipated power has its maximum at omega0 for this force input.
    bw_low = np.sqrt(omega0**2 + gamma**2) - gamma
    bw_high = np.sqrt(omega0**2 + gamma**2) + gamma
    bandwidth = bw_high - bw_low
    omega_d = np.unique(np.r_[np.linspace(0, 2.5 * omega0, 2000),
                               bw_low, omega0, bw_high])
    denominator = (omega0**2 - omega_d**2)**2 + (2 * gamma * omega_d)**2
    power_norm = (2 * gamma * omega_d)**2 / denominator

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), layout='constrained')
    ax1.plot(omega_d / omega0, power_norm, 'b-', lw=2.5, label='Mean power / peak power')
    ax1.fill_between(omega_d / omega0, 0, power_norm,
                     where=(omega_d >= bw_low) & (omega_d <= bw_high), alpha=0.2)
    ax1.axhline(0.5, color='green', ls=':', label='Half power')
    ax1.plot([bw_low / omega0, bw_high / omega0], [0.5, 0.5], 'ro')
    ax1.set(xlabel=r'$\Omega/\omega_0$', ylabel='Normalized mean power',
            xlim=(0, 2.5), ylim=(0, 1.1))
    ax1.set_title(f'Power Bandwidth: Q = {Q:.1f}\nDelta omega = {bandwidth:.3f} rad/s')
    ax1.legend(loc='upper center', bbox_to_anchor=(0.5, -0.20), fontsize=9)

    # Exact free response released from rest: x(0)=1 m, v(0)=0.
    t_free = np.linspace(0, 10 * 2 * np.pi / omega0, 2000)
    sol = odeint(lambda state, t: [state[1], -2 * gamma * state[1] - omega0**2 * state[0]],
                 [1.0, 0.0], t_free)
    energy_norm = (sol[:, 1]**2 + omega0**2 * sol[:, 0]**2) / omega0**2
    reference_periods = t_free * omega0 / (2 * np.pi)
    ax2.plot(reference_periods, energy_norm, 'purple', lw=2, label='Exact E(t) / E(0)')
    ax2.plot(reference_periods, np.exp(-2 * gamma * t_free), 'k--', alpha=0.6,
             label='Exponential guide (weak damping)')
    ax2.axhline(1 / np.e, color='orange', ls=':', label='1/e level')
    ax2.set(xlabel='t / T0 (reference periods)', ylabel='E / E(0)', ylim=(-0.02, 1.05))
    ax2.set_title('Free Response: Energy from x and v\nT0 = 2 pi / omega0; not always an oscillation')
    ax2.legend(loc='upper center', bbox_to_anchor=(0.5, -0.20), fontsize=9)
    plt.show()
    print(f'Q = {Q:.2f}; gamma = {gamma:.3f} s^-1')
    print(f'Half-power frequencies: {bw_low:.3f}, {bw_high:.3f} rad/s')
    print(f'omega0 / bandwidth = {omega0 / bandwidth:.2f}')

physics_interact(interactive_Q_factor,
    Q=widgets.FloatSlider(value=5.0, min=0.5, max=50.0, step=0.5,
                          description='Q factor:',
                          style={'description_width': 'initial'})
);

### Engineering Application: Vibration Dampers

In engineering, vibration dampers (also called tuned mass dampers) protect structures from resonance. The Taipei 101 skyscraper has a 730-ton steel pendulum that swings to counteract wind-induced oscillations.

#### The Design Problem

Given a structure with natural frequency $\omega_0$ that is subject to a periodic driving force (e.g., wind, machinery, earthquakes), you need to choose a damping coefficient $b$ to keep the vibration amplitude below a safe limit.

**Trade-offs:**
- Too little damping: large resonance amplitude (dangerous!)
- Too much damping: system responds slowly, poor vibration isolation at high frequencies
- Optimal: balance between peak reduction and broadband isolation

#### ⏱️ Checkpoint 2 of 3 — Think · Pair · Explain

**Small class activity / Kısa sınıf etkinliği.** A measured mean-power resonance has $\omega_0=12\,\mathrm{rad/s}$ and half-power limits $10.6$ and $13.6\,\mathrm{rad/s}$.

**Think — 1 minute:** Identify the subtraction needed for bandwidth; explain why Q has no unit.

**Pair — 2 minutes:** Compare your sign, units and first equation. Explain a disagreement before checking the model response.

**Explain — 2 minutes:** Write the physical reason for your answer; the calculation alone is not the explanation.

**Worked model response / Adım adım örnek yanıt**

$$\Delta\omega=13.6-10.6=3.00\,\mathrm{rad/s},\qquad
Q=\frac{\omega_0}{\Delta\omega}=\frac{12}{3}=4.$$
The associated decay rate is $\gamma=\Delta\omega/2=1.5\,\mathrm{s^{-1}}$. Subtract the frequency coordinates of the two half-power crossings, not their heights.

**Türkçe:** Yatay eksendeki iki frekansı çıkarıyoruz. Her ikisi de aynı birimdedir. $Q$ hesaplanırken frekans birimleri sadeleşir. Grafiğin genlik mi güç mü gösterdiğini okumadan bant genişliği tanımını kullanma.

**Your explanation / Açıklaman:** write or discuss your reasoning in words, with an equation or sketch where useful.


### Interactive: Vibration Damper Design Tool

Design a vibration damper for a machine base. Adjust parameters and see if the maximum vibration stays below the safety limit.

In [ ]:
#@title Run the demonstration — predict, adjust, observe
def vibration_damper_tool(m, k, b, F0, rpm_min, rpm_max, x_safe):
    """Interactive vibration damper design tool."""
    omega0 = np.sqrt(k / m)
    gamma = b / (2 * m)
    f0 = F0 / m
    Q = omega0 / (2 * gamma) if gamma > 0 else float('inf')

    if rpm_min > rpm_max:
        print("RPM bounds were reversed; the lower and upper values are sorted below.")
    rpm_min, rpm_max = sorted((rpm_min, rpm_max))
    # Convert RPM range to angular frequency
    omega_min = rpm_min * 2 * np.pi / 60
    omega_max = rpm_max * 2 * np.pi / 60
    omega_peak = np.sqrt(max(0, omega0**2 - 2 * gamma**2))
    # Include the exact candidate maximum; a coarse grid can miss a narrow peak.
    omega_d = np.unique(np.r_[np.linspace(omega_min, omega_max, 1000),
                               np.clip(omega_peak, omega_min, omega_max)])

    A_resp = (F0 / k) / np.sqrt((1 - (omega_d / omega0)**2)**2 + (2 * gamma * omega_d / omega0**2)**2)

    # Full range for reference
    omega_full = np.unique(np.r_[np.linspace(0, max(omega_max * 1.5, 1.3 * omega0), 1000), omega_peak])
    A_full = (F0 / k) / np.sqrt((1 - (omega_full / omega0)**2)**2 + (2 * gamma * omega_full / omega0**2)**2)
    rpm_full = omega_full * 60 / (2 * np.pi)

    rpm_range = omega_d * 60 / (2 * np.pi)
    A_max = np.max(A_resp)
    safe = A_max <= x_safe

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), layout="constrained")

    # --- Amplitude vs RPM ---
    ax1.plot(rpm_full, A_full * 1000, 'b-', lw=2, label='Response')
    ax1.axhspan(0, x_safe * 1000, alpha=0.1, color='green', label='Safe zone')
    ax1.axhline(y=x_safe * 1000, color='green', ls='--', lw=2)
    ax1.axvspan(rpm_min, rpm_max, alpha=0.15, color='orange', label='Operating range')
    ax1.axhline(y=A_max * 1000, color='red' if not safe else 'green', ls=':', lw=2,
                label=f'Max in range: {A_max*1000:.2f} mm')

    f0_hz = omega0 / (2 * np.pi)
    rpm_resonance = f0_hz * 60
    ax1.axvline(x=rpm_resonance, color='gray', ls='--', alpha=0.5,
                label=f'Natural frequency: {rpm_resonance:.0f} RPM')

    ax1.set_xlabel('Frequency (RPM)', fontsize=12)
    ax1.set_ylabel('Amplitude (mm)', fontsize=12)
    status = 'SAFE' if safe else 'EXCEEDS LIMIT'
    status_color = 'green' if safe else 'red'
    ax1.set_title(f'Vibration Response — {status}', fontweight='bold',
                  fontsize=14, color=status_color)
    ax1.legend(loc='upper center', bbox_to_anchor=(0.5, -0.20), ncol=2, fontsize=9)

    # --- Parameter summary ---
    ax2.axis('off')
    summary = [
        ['Parameter', 'Value'],
        ['Mass m', f'{m:.1f} kg'],
        ['Stiffness k', f'{k:.0f} N/m'],
        ['Damping b', f'{b:.1f} N s/m'],
        [r'Natural freq $\omega_0$', f'{omega0:.2f} rad/s ({omega0/(2*np.pi):.2f} Hz)'],
        [r'Decay rate $\gamma$', f'{gamma:.3f} s$^{{-1}}$'],
        ['Q factor', f'{Q:.1f}'],
        ['Driving force $F_0$', f'{F0:.1f} N'],
        ['Operating range', f'{rpm_min:.0f} - {rpm_max:.0f} RPM'],
        ['Max amplitude in range', f'{A_max*1000:.2f} mm'],
        ['Safety limit', f'{x_safe*1000:.1f} mm'],
        ['Status', status],
    ]

    table = ax2.table(cellText=[[r[0], r[1]] for r in summary],
                      colWidths=[0.48, 0.52], loc='center', cellLoc='left')
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1, 1.8)

    # Color the status row
    table[len(summary) - 1, 1].set_facecolor('lightgreen' if safe else 'lightcoral')
    table[0, 0].set_facecolor('lightblue')
    table[0, 1].set_facecolor('lightblue')

    ax2.set_title('Design Parameters', fontweight='bold', fontsize=14)

    plt.show()

physics_interact(vibration_damper_tool,
    m=widgets.FloatSlider(value=50.0, min=5.0, max=200.0, step=5.0,
                          description='Mass m (kg):', style={'description_width': 'initial'}),
    k=widgets.FloatSlider(value=5000.0, min=500.0, max=50000.0, step=500.0,
                          description='Stiffness k (N/m):', style={'description_width': 'initial'}),
    b=widgets.FloatSlider(value=100.0, min=1.0, max=2000.0, step=10.0,
                          description='Damping b (N s/m):', style={'description_width': 'initial'}),
    F0=widgets.FloatSlider(value=50.0, min=5.0, max=500.0, step=5.0,
                           description='Force F0 (N):', style={'description_width': 'initial'}),
    rpm_min=widgets.FloatSlider(value=300, min=0, max=3000, step=50,
                                description='RPM min:', style={'description_width': 'initial'}),
    rpm_max=widgets.FloatSlider(value=1800, min=100, max=5000, step=50,
                                description='RPM max:', style={'description_width': 'initial'}),
    x_safe=widgets.FloatSlider(value=0.005, min=0.001, max=0.05, step=0.001,
                               description='Safety limit (m):', style={'description_width': 'initial'},
                               readout_format='.3f')
);

### Worked Examples

#### Example 1: Resonance of a Car on a Bumpy Road

A car (1500 kg, suspension $k = 60000\,\mathrm{N/m}$, $b = 3000\,\mathrm{N\,s/m}$) drives over speed bumps spaced 10 m apart. At what speed does the car experience resonance? What is the Q factor?

**1. Find the natural angular frequency, then convert to cycles per second.**

$$\omega_0=\sqrt{\frac{k}{m}}=\sqrt{\frac{60000}{1500}}
=\sqrt{40}=6.32\,\mathrm{rad/s},$$
$$f_0=\frac{\omega_0}{2\pi}=1.01\,\mathrm{Hz}.$$

**2. Convert bump spacing into a driving frequency.** With spacing $d=10\,\mathrm m$, one cycle takes $d/v$, so $f=v/d$. Match this to $f_0$ for the simple natural-frequency estimate:

$$v\approx df_0=10(1.01)=10.1\,\mathrm{m/s}.$$

Convert the units explicitly:

$$10.1\frac{\mathrm m}{\mathrm s}
\times\frac{1\,\mathrm{km}}{1000\,\mathrm m}
\times\frac{3600\,\mathrm s}{1\,\mathrm h}
=36.2\,\mathrm{km/h}.$$

**3. Calculate damping and quality factor.**

$$\gamma=\frac{3000}{2(1500)}=1.00\,\mathrm{s^{-1}},\qquad
Q=\frac{\omega_0}{2\gamma}=\frac{6.32}{2}=3.16.$$

**What “maximum” means:** A constant-amplitude *force* gives its displacement peak at $\Omega_r=\sqrt{\omega_0^2-2\gamma^2}=\sqrt{38}\,\mathrm{rad/s}$, corresponding to 9.81 m/s. For a sinusoidal road-height input and absolute car-body displacement, $r_{\rm peak}^2=\dfrac{2}{1+\sqrt{1+8\zeta^2}}$ gives 9.83 m/s. The 10.1 m/s result is a useful estimate, not an exact maximum for either damped model. **TR:** Rezonans hızını söylerken girişin kuvvet mi yol yüksekliği mi olduğunu belirt.


In [ ]:
#@title Optional numerical check — the algebra is explained above
m_car = 1500     # kg
k_car = 60000    # N/m
b_car = 3000     # N·s/m
bump_spacing = 10  # m

omega0 = np.sqrt(k_car / m_car)
gamma = b_car / (2 * m_car)
f0_hz = omega0 / (2 * np.pi)
Q = omega0 / (2 * gamma)

# Natural-frequency match; exact displacement peaks depend on the drive model.
v_resonance = f0_hz * bump_spacing
v_resonance_kmh = v_resonance * 3.6

print("=" * 55)
print("Example 1: Car on Bumpy Road")
print("=" * 55)
print(f"Natural frequency: omega_0 = {omega0:.2f} rad/s")
print(f"                  f_0 = {f0_hz:.2f} Hz")
print(f"Damping: gamma = {gamma:.2f} s^-1")
print(f"Q factor: {Q:.2f}")
print(f"\nFrequency-match estimate: v = f_0 * d = {f0_hz:.2f} * {bump_spacing}")
print(f"                 v = {v_resonance:.1f} m/s = {v_resonance_kmh:.0f} km/h")
omega_force_peak = np.sqrt(omega0**2 - 2 * gamma**2)
v_force_peak = bump_spacing * omega_force_peak / (2 * np.pi)
zeta = gamma / omega0
r_base_peak = np.sqrt(2 / (1 + np.sqrt(1 + 8 * zeta**2)))
v_base_peak = bump_spacing * omega0 * r_base_peak / (2 * np.pi)
print("An exact displacement maximum depends on how the road drives the car.")
print(f"Constant-amplitude force model peak: {v_force_peak:.3f} m/s")
print(f"Sinusoidal road-height input, body displacement peak: {v_base_peak:.3f} m/s")

#### Example 2: Steady-State Amplitude

A 2 kg mass on a spring ($k = 200\,\mathrm{N/m}$) with damping $b = 4\,\mathrm{N\,s/m}$ is driven by $F(t) = 10\cos(8t)\,\mathrm{N}$. Find the steady-state amplitude and phase lag.

**Calculate four small pieces.** From $10\cos(8t)$, read $F_0=10\,\mathrm{N}$ and $\Omega=8\,\mathrm{rad/s}$. Use the force-form denominator:
$$u=k-m\Omega^2=200-2(8^2)=200-128=72\ \mathrm{N/m},$$
$$v=b\Omega=4(8)=32\ \mathrm{N/m},\qquad D=\sqrt{72^2+32^2}=\sqrt{6208}=78.8\ \mathrm{N/m}.$$
**Divide force amplitude by the denominator.**

$$A=\frac{F_0}{D}=\frac{10}{78.8}=0.127\,\mathrm m\approx12.7\,\mathrm{cm}.$$

**Determine the phase quadrant before taking an inverse tangent.** Both $u$ and $v$ are positive, so

$$\delta=\tan^{-1}\!\left(\frac{32}{72}\right)=24.0^\circ.$$

If $u$ were negative, this principal inverse-tangent value alone would choose the wrong quadrant; use the full two-argument phase relation given earlier.

**Interpretation:** The natural frequency is $\sqrt{200/2}=10\,\mathrm{rad/s}$. Driving at 8 rad/s is below it, so the lag is below $90^\circ$. The static displacement would be $F_0/k=0.050\,\mathrm{m}$; the dynamic amplitude is 2.54 times larger. **TR:** Uzun formülü tek işlemde yazmak yerine paydadaki iki terimi ayrı hesapla. Aşağıdaki kod sonucu doğrular.


In [ ]:
#@title Optional numerical check — the algebra is explained above
m = 2.0; k = 200.0; b = 4.0; F0 = 10.0; omega_d = 8.0

omega0 = np.sqrt(k / m)
gamma = b / (2 * m)
f0 = F0 / m

A_ss = f0 / np.sqrt((omega0**2 - omega_d**2)**2 + (2 * gamma * omega_d)**2)
delta = np.arctan2(2 * gamma * omega_d, omega0**2 - omega_d**2)
A_static = F0 / k

print("=" * 55)
print("Example 2: Steady-State Amplitude")
print("=" * 55)
print(f"omega_0 = sqrt({k}/{m}) = {omega0:.2f} rad/s")
print(f"gamma = {b}/(2*{m}) = {gamma:.2f} s^-1")
print(f"omega_d = {omega_d:.2f} rad/s")
print(f"\nSteady-state amplitude: A = {A_ss:.4f} m = {A_ss*1000:.2f} mm")
print(f"Phase lag: delta = {np.degrees(delta):.1f} degrees")
print(f"Static deflection: x_static = F0/k = {A_static:.4f} m")
print(f"Amplification factor: A/x_static = {A_ss/A_static:.2f}")

# Verify with odeint
def driven_ode(state, t):
    return [state[1], f0 * np.cos(omega_d * t) - 2 * gamma * state[1] - omega0**2 * state[0]]

t = np.linspace(0, 30, 5000)
sol = odeint(driven_ode, [0, 0], t)
A_numerical = np.max(np.abs(sol[-2000:, 0]))
print(f"\nNumerical verification (last portion): A_max = {A_numerical:.4f} m")

#### Example 3: Q Factor and Bandwidth

An RLC circuit has a resonant frequency of 1000 Hz and a bandwidth of 20 Hz. Find the Q factor. If the resistance is doubled, what is the new bandwidth?

**Series-RLC current/power-response assumption.** The two frequencies must use the same unit:

$$Q=\frac{f_0}{\Delta f}=\frac{1000\,\mathrm{Hz}}{20\,\mathrm{Hz}}=50.$$

In this model $\Delta f=R/(2\pi L)$. Keeping $L,C$ fixed and replacing $R$ by $2R$ gives

$$\Delta f_{\mathrm{new}}=\frac{2R}{2\pi L}=2\Delta f=40\,\mathrm{Hz},$$
$$Q_{\mathrm{new}}=\frac{1000}{40}=25.$$

The resonant frequency $f_0=1/(2\pi\sqrt{LC})$ stays unchanged. The bandwidth doubles and the quality factor halves.

**TR:** Önce devre modelini belirt; direnç artışı daha fazla enerji kaybı ve daha geniş tepe verir. Devre bilgisi bu haftanın mekanik temel hedefleri için zorunlu değildir.

<a id="x12-problems"></a>

## 3. Problem set — predict, then check / Problem seti

Reach the symbolic answer and check a limiting case before opening the **Answer**.


### Core problems (L1) / Temel problemler

Everyone should complete these; they follow the worked examples directly.

#### P1  ·  L1
A damped oscillator has natural frequency $\omega_0 = 10\,\mathrm{rad/s}$ and damping $\gamma = 0.5\,\mathrm{s^{-1}}$. Calculate the quality factor $Q$.

**Türkçe — problem:** Doğal açısal frekansı $\omega_0=10\,\mathrm{rad/s}$ ve sönüm hızı $\gamma=0.5\,\mathrm{s^{-1}}$ olan osilatörün kalite faktörü $Q$’yu bulun.

<details><summary>Answer</summary>

$Q = 10$


</details>

**Your working / Çözümün:** givens with units → diagram → principle → algebra → substitution → answer and check. Use paper or add a text cell.  
*Full worked solution:* Module 12 P1 in the solutions collection (file `Week_12_Python_Solutions.ipynb`, opens 18 December 2026).

#### P2  ·  L1
A $3.0\,\mathrm{kg}$ mass on a spring ($k = 300\,\mathrm{N/m}$) is driven at the natural frequency with a force amplitude $F_0 = 6.0\,\mathrm{N}$. The damping coefficient is $b = 3.0\,\mathrm{N\,s/m}$. Find the steady-state amplitude at resonance.

**Türkçe — problem:** $k=300\,\mathrm{N/m}$ olan yaya bağlı $3.0\,\mathrm{kg}$ kütle, doğal frekansında $F_0=6.0\,\mathrm N$ genlikli kuvvetle sürülür. Sönüm katsayısı $b=3.0\,\mathrm{N\,s/m}$’dir. Bu sürme frekansındaki kararlı durum genliğini bulun.

<details><summary>Answer</summary>

$A = 0.200\,\mathrm{m}$


</details>

**Your working / Çözümün:** givens with units → diagram → principle → algebra → substitution → answer and check. Use paper or add a text cell.  
*Full worked solution:* Module 12 P2 in the solutions collection (file `Week_12_Python_Solutions.ipynb`, opens 18 December 2026).

#### P3  ·  L1
A forced oscillator has $\omega_0 = 20\,\mathrm{rad/s}$. It is driven at $\omega_d = 15\,\mathrm{rad/s}$ with $\gamma = 2.0\,\mathrm{s^{-1}}$. Calculate the phase lag $\delta$ between the driving force and the response.

**Türkçe — problem:** Zorlanmış bir osilatörde $\omega_0=20\,\mathrm{rad/s}$, sürme frekansı $\omega_d=15\,\mathrm{rad/s}$ ve $\gamma=2.0\,\mathrm{s^{-1}}$’dir. Sürme kuvveti ile yer değiştirme arasındaki faz gecikmesini $\delta$ bulun.

<details><summary>Answer</summary>

$$\delta=\tan^{-1}[60/175]=18.9^\circ=0.330$$

rad. The earlier 19.3° value was inaccurate; the lag is below 90° because driving is below the natural frequency.


</details>

**Your working / Çözümün:** givens with units → diagram → principle → algebra → substitution → answer and check. Use paper or add a text cell.  
*Full worked solution:* Module 12 P3 in the solutions collection (file `Week_12_Python_Solutions.ipynb`, opens 18 December 2026).


#### P4  ·  L1
A driven harmonic oscillator reaches steady state with amplitude $A = 0.050\,\mathrm{m}$ at driving frequency $\omega_d = 25\,\mathrm{rad/s}$. The damping is $\gamma = 1.5\,\mathrm{s^{-1}}$. What is the average power delivered to the oscillator? (Use $\langle P \rangle = m\gamma\omega_d^2 A^2$.  Take $m = 2.0\,\mathrm{kg}$.)

**Türkçe — problem:** Bir osilatör $\omega_d=25\,\mathrm{rad/s}$ sürme frekansında $A=0.050\,\mathrm m$ kararlı durum genliğine ulaşır. $\gamma=1.5\,\mathrm{s^{-1}}$ ve $m=2.0\,\mathrm{kg}$ için ortalama aktarılan gücü bulun. Verilen bağıntıyı kullanın: $\langle P\rangle=m\gamma\omega_d^2A^2$.

<details><summary>Answer</summary>

$$\langle P \rangle = 4.69\,\mathrm{W}$$


</details>

**Your working / Çözümün:** givens with units → diagram → principle → algebra → substitution → answer and check. Use paper or add a text cell.  
*Full worked solution:* Module 12 P4 in the solutions collection (file `Week_12_Python_Solutions.ipynb`, opens 18 December 2026).

### Pause and explain / Dur ve açıkla

Before the intermediate problems, explain one core result to a partner.

#### ⏱️ Checkpoint 3 of 3 — Think · Pair · Explain

**Small class activity / Kısa sınıf etkinliği.** A shaft makes $600$ revolutions per minute. An analyst types “600” into a control labelled rad/s.

**Think — 1 minute:** Convert the speed in two explicit steps and diagnose the mistake.

**Pair — 2 minutes:** Compare your sign, units and first equation. Explain a disagreement before checking the model response.

**Explain — 2 minutes:** Write the physical reason for your answer; the calculation alone is not the explanation.

**Worked model response / Adım adım örnek yanıt**

$$f=600\frac{\mathrm{rev}}{\mathrm{min}}\frac{1\,\mathrm{min}}{60\,\mathrm s}
=10\,\mathrm{Hz},\qquad \Omega=2\pi f=62.8\,\mathrm{rad/s}.$$
The value 600 rad/s describes a different drive, about 9.55 times the intended angular frequency. Both the machine model and the software display must use the same quantity and unit.

**Türkçe:** Dakikayı saniyeye çevirmek için 60’a böldük; çevrimi radyana çevirmek için $2\pi$ ile çarptık. Birim hatası rezonansın yanlış yerde görünmesine neden olabilir. Sayıyı değiştirmeden yalnızca etiketini değiştirmek birim dönüşümü değildir.

**Your explanation / Açıklaman:** write or discuss your reasoning in words, with an equation or sketch where useful.


### Intermediate problems (L2) / Orta düzey

Combine two ideas from this week.

#### P5  ·  L2
A machine base ($m = 80\,\mathrm{kg}$) is mounted on springs with total stiffness $k = 32{,}000\,\mathrm{N/m}$ and total damping $b = 640\,\mathrm{N\,s/m}$. The machine generates a sinusoidal force $F_0 = 200\,\mathrm{N}$ at $1200\,\mathrm{RPM}$. Find (a) the natural frequency in Hz, (b) the quality factor, (c) the steady-state vibration amplitude, and (d) whether the amplitude exceeds a safety limit of $0.5\,\mathrm{mm}$.

**Türkçe — problem:** $m=80\,\mathrm{kg}$ makine tabanı, toplam $k=32{,}000\,\mathrm{N/m}$ yay sertliği ve $b=640\,\mathrm{N\,s/m}$ sönümle taşınır. Makine $1200\,\mathrm{RPM}$’de $F_0=200\,\mathrm N$ sinüzoidal kuvvet üretir. (a) Doğal frekansı Hz cinsinden, (b) kalite faktörünü, (c) kararlı titreşim genliğini bulun; (d) bu genliğin $0.5\,\mathrm{mm}$ sınırını aşıp aşmadığını belirleyin.

<details><summary>Answer</summary>

**(a) Natural frequency.**

$$\omega_0=\sqrt{\frac{k}{m}}=20\,\mathrm{rad/s},\qquad
f_0=\frac{\omega_0}{2\pi}=\frac{20}{2\pi}=3.18\,\mathrm{Hz}.$$

**(b) Damping and quality factor.**

$$\gamma=\frac{b}{2m}=4.0\,\mathrm{s^{-1}},\qquad
Q=\frac{\omega_0}{2\gamma}=2.5.$$

**(c) Convert the drive before substituting.** The given $1200\,\mathrm{RPM}$ is $20\,\mathrm{Hz}$, so $\omega_d=40\pi\,\mathrm{rad/s}$. Then

$$A=\frac{F_0/m}{\sqrt{(\omega_0^2-\omega_d^2)^2+(2\gamma\omega_d)^2}}
\approx\frac{2.5}{15424}\,\mathrm m=0.162\,\mathrm{mm}.$$

**(d)** No: $0.162\,\mathrm{mm}<0.5\,\mathrm{mm}$, so the amplitude is within the stated limit.

**[CORRECTED]** Earlier values $Q=5.0$ and $A=0.040\,\mathrm{mm}$ were incorrect; the conclusion in part **(d)** is unchanged.


</details>

**Your working / Çözümün:** givens with units → diagram → principle → algebra → substitution → answer and check. Use paper or add a text cell.  
*Full worked solution:* Module 12 P5 in the solutions collection (file `Week_12_Python_Solutions.ipynb`, opens 18 December 2026).

#### P6  ·  L2
A car ($1400\,\mathrm{kg}$, suspension $k = 55{,}000\,\mathrm{N/m}$, $b = 4000\,\mathrm{N\,s/m}$) drives over sinusoidal speed bumps spaced $8.0\,\mathrm{m}$ apart. (a) At what speed does the car experience maximum bouncing (resonance)? (b) What is the Q factor of the suspension? (c) Is the system underdamped or overdamped?

**Türkçe — problem:** $1400\,\mathrm{kg}$ bir otomobilin süspansiyonu $k=55{,}000\,\mathrm{N/m}$ ve $b=4000\,\mathrm{N\,s/m}$ ile modellenir. Araç aralarında $8.0\,\mathrm m$ olan sinüzoidal yol tümseklerinden geçer. (a) En büyük düşey titreşimin görüldüğü hızı, (b) kalite faktörünü ve (c) sönüm rejimini bulun. “En büyük” yanıtı için giriş ve ölçülen tepki türünü açıkça belirtin.

<details><summary>Answer</summary>

**(a)** Natural-frequency estimate: $v\approx7.98045\,\mathrm{m/s}$ $=28.7\,\mathrm{km/h}$. For constant-amplitude road displacement and absolute body displacement, the actual peak is $7.63\,\mathrm{m/s}$ $=27.5\,\mathrm{km/h}$; a constant-amplitude force model gives $7.55\,\mathrm{m/s}$ $=27.2\,\mathrm{km/h}$. Specify which model is meant by “maximum bouncing.” **(b)** $Q=2.19$. **(c)** Underdamped ($\zeta=0.228<1$).


</details>

**Your working / Çözümün:** givens with units → diagram → principle → algebra → substitution → answer and check. Use paper or add a text cell.  
*Full worked solution:* Module 12 P6 in the solutions collection (file `Week_12_Python_Solutions.ipynb`, opens 18 December 2026).


#### P7  ·  L2
An RLC circuit has $L = 0.10\,\mathrm{H}$, $C = 10$ $\mu$F, and $R = 20$ $\Omega$. (a) Find the resonant frequency $f_0$. (b) Find the quality factor. (c) What is the bandwidth $\Delta f$? (d) If $R$ is reduced to $5$ $\Omega$, what happens to $Q$ and the bandwidth?

**Türkçe — problem:** Bir RLC devresinde $L=0.10\,\mathrm H$, $C=10\,\mu\mathrm F$ ve $R=20\,\Omega$’dur. (a) Rezonans frekansını $f_0$, (b) kalite faktörünü, (c) bant genişliğini $\Delta f$ bulun. (d) Direnç $5\,\Omega$ yapılırsa $Q$ ve bant genişliği nasıl değişir?

<details><summary>Answer</summary>

**(a)** $f_0 = 159\,\mathrm{Hz}$; **(b)** $Q = 5.0$; **(c)** $\Delta f = 31.8\,\mathrm{Hz}$; **(d)** $Q = 20$, $\Delta f = 7.96\,\mathrm{Hz}$


</details>

**Your working / Çözümün:** givens with units → diagram → principle → algebra → substitution → answer and check. Use paper or add a text cell.  
*Full worked solution:* Module 12 P7 in the solutions collection (file `Week_12_Python_Solutions.ipynb`, opens 18 December 2026).


#### P8  ·  L2
A $0.50\,\mathrm{kg}$ mass on a spring ($k = 50\,\mathrm{N/m}$) with $b = 1.0\,\mathrm{N\,s/m}$ is driven by $F(t) = 3.0\cos(\omega_d t)\,\mathrm{N}$. Find the steady-state amplitude for (a) $\omega_d = 5\,\mathrm{rad/s}$, (b) $\omega_d = 10\,\mathrm{rad/s}$ (at resonance), and (c) $\omega_d = 15\,\mathrm{rad/s}$.

**Türkçe — problem:** $m=0.50\,\mathrm{kg}$, $k=50\,\mathrm{N/m}$ ve $b=1.0\,\mathrm{N\,s/m}$ olan sistem $F(t)=(3.0\,\mathrm N)\cos(\omega_dt)$ kuvvetiyle sürülür. (a) $\omega_d=5\,\mathrm{rad/s}$, (b) $10\,\mathrm{rad/s}$ ve (c) $15\,\mathrm{rad/s}$ için kararlı durum genliklerini bulun.

<details><summary>Answer</summary>

Use $\omega_0=10\,\mathrm{rad/s}$ and $\gamma=1.0\,\mathrm{s^{-1}}$:

$$A=\frac{F_0/m}{\sqrt{(\omega_0^2-\omega_d^2)^2+(2\gamma\omega_d)^2}}.$$

<table width="100%">
<thead>
<tr>
<th align="left" width="144" scope="col">Case</th>
<th align="left" width="264" scope="col">Denominator calculation in $\mathrm{s^{-2}}$</th>
<th align="left" width="208" scope="col">Amplitude</th>
</tr>
</thead>
<tbody>
<tr>
<td><strong>(a)</strong> $\omega_d=5\,\mathrm{rad/s}$</td>
<td>$\sqrt{(100-25)^2+10^2}=\sqrt{5725}$</td>
<td>$A=\dfrac{6}{\sqrt{5725}}\,\mathrm m=0.0793\,\mathrm m$</td>
</tr>
<tr>
<td><strong>(b)</strong> $\omega_d=10\,\mathrm{rad/s}$</td>
<td>$\sqrt{0^2+20^2}=20$</td>
<td>$A=\dfrac{6}{20}\,\mathrm m=0.300\,\mathrm m$</td>
</tr>
<tr>
<td><strong>(c)</strong> $\omega_d=15\,\mathrm{rad/s}$</td>
<td>$\sqrt{(100-225)^2+30^2}=\sqrt{16525}$</td>
<td>$A=\dfrac{6}{\sqrt{16525}}\,\mathrm m=0.0467\,\mathrm m$</td>
</tr>
</tbody>
</table>

**[CORRECTED]** The earlier $0.0800\,\mathrm m$ and $0.0480\,\mathrm m$ for **(a)** and **(c)** dropped the damping term $(2\gamma\omega_d)^2$. Use the same complete denominator in all three cases. The actual displacement peak is at

$$\sqrt{\omega_0^2-2\gamma^2}=9.90\,\mathrm{rad/s},$$

slightly below $\omega_0$.


</details>

**Your working / Çözümün:** givens with units → diagram → principle → algebra → substitution → answer and check. Use paper or add a text cell.  
*Full worked solution:* Module 12 P8 in the solutions collection (file `Week_12_Python_Solutions.ipynb`, opens 18 December 2026).


### Challenge problems (L3) / İleri düzey

Engineering-style problems with several steps; useful preparation for the exams.

#### P9  ·  L3
A sensitive optical table ($m = 500\,\mathrm{kg}$) must be isolated from floor vibrations in the range $5$--$50\,\mathrm{Hz}$. The table is mounted on pneumatic isolators modeled as springs with adjustable damping. (a) What spring constant $k$ is needed so that the natural frequency is $2.0\,\mathrm{Hz}$ (below the vibration range)? (b) What damping ratio $\zeta = \gamma/\omega_0$ should be chosen so the transmissibility $T = A_\text{table}/A_\text{floor} < 0.05$ at $5\,\mathrm{Hz}$? (c) What is $T$ at $50\,\mathrm{Hz}$ with this damping?

**Türkçe — problem:** $500\,\mathrm{kg}$ hassas optik masa, $5$–$50\,\mathrm{Hz}$ taban titreşimlerinden yalıtılacaktır. Ayarlanabilir sönümlü pnömatik izolatörler yaylarla modellenir. (a) Doğal frekans $2.0\,\mathrm{Hz}$ olacak şekilde $k$’yı bulun. (b) $5\,\mathrm{Hz}$’de geçirgenlik $T=A_{\rm masa}/A_{\rm taban}<0.05$ olsun diye $\zeta=\gamma/\omega_0$ seçin. (c) Bu sönümle $50\,\mathrm{Hz}$’de $T$ kaçtır? Koşulların birlikte sağlanabilir olup olmadığını da kontrol edin.

<details><summary>Answer</summary>

**(a) Spring stiffness.**

$$k=m\omega_0^2=500(4\pi)^2=79000\,\mathrm{N/m}.$$

**(b) Check feasibility before choosing damping.** At $5\,\mathrm{Hz}$ the frequency ratio is $r=5/2=2.5$. For base-displacement input, both sides of the transmissibility inequality are nonnegative, so squaring is valid:

$$T^2=\frac{1+25\zeta^2}{(1-6.25)^2+25\zeta^2}<0.05^2.$$

Multiply by the positive denominator and collect the terms:

$$0.931+24.9\zeta^2<0.$$

This is impossible for any real damping ratio. The smallest possible transmissibility is $T=1/5.25=0.190$ at zero damping.

**(c)** There is no unique answer because part **(b)** has no solution. In general,

$$T(50\,\mathrm{Hz})=\sqrt{\frac{1+2500\zeta^2}{389376+2500\zeta^2}}.$$

The zero-damping lower-bound example is $1/624=0.00160$; it does **not** solve **(b)**. Even zero damping would require $f_0<1.09\,\mathrm{Hz}$ for the strict $5\,\mathrm{Hz}$ target.


</details>

**Your working / Çözümün:** givens with units → diagram → principle → algebra → substitution → answer and check. Use paper or add a text cell.  
*Full worked solution:* Module 12 P9 in the solutions collection (file `Week_12_Python_Solutions.ipynb`, opens 18 December 2026).


#### P10  ·  L3
A tuned mass damper (TMD) is to be designed for a bridge deck that resonates dangerously at $f_0 = 1.8\,\mathrm{Hz}$ under pedestrian loading. The bridge effective mass is $M = 50{,}000\,\mathrm{kg}$, stiffness $K = 6.40 \times 10^6\,\mathrm{N/m}$, and structural damping $B = 5000\,\mathrm{N\,s/m}$. The TMD consists of a secondary mass $m_d = 2500\,\mathrm{kg}$ (5% of bridge mass) on a spring $k_d$ with damper $b_d$. (a) What spring constant $k_d$ should be used so the TMD natural frequency matches the bridge? (b) For optimal damping, use $\zeta_d = \sqrt{3\mu/8}$ where $\mu = m_d/M$. Find $b_d$. (c) Estimate the reduction factor in peak response amplitude.

**Türkçe — problem:** Yaya yüklemesi altında $f_0=1.8\,\mathrm{Hz}$’de rezonansa giren köprüye ayarlı kütle sönümleyici tasarlanır. Köprünün etkin kütlesi $M=50{,}000\,\mathrm{kg}$, sertliği $K=6.40\times10^6\,\mathrm{N/m}$ ve sönümü $B=5000\,\mathrm{N\,s/m}$’dir. Ek kütle $m_d=2500\,\mathrm{kg}$, yay $k_d$ ve sönümleyici $b_d$ kullanılır. (a) Doğal frekanslar eşleşecek şekilde $k_d$’yi, (b) $\mu=m_d/M$ ve $\zeta_d=\sqrt{3\mu/8}$ ile $b_d$’yi bulun. (c) Tepe tepki genliğinin azalma oranını, kullanılan kuvvet ve tepki modeliyle birlikte tahmin edin.

<details><summary>Answer</summary>

**(a)**

$$k_d=2500(2\pi\times1.8)^2=3.20\times10^5\,\mathrm{N/m}.$$

**(b)** $\zeta_d=\sqrt{3(0.05)/8}=0.137$,

$$b_d=2\zeta_dm_d\omega_t=7740\,\mathrm{N\,s/m}.$$

**(c)** Under a constant-amplitude harmonic force on the bridge, compare peak bridge displacement across frequency using the two-mass equations: the peak ratio with/without TMD is about 0.0731 (92.7% reduction, 13.7 times smaller). The response measure and forcing assumptions must be stated; the earlier 70% estimate was unsupported by the supplied model.


</details>

**Your working / Çözümün:** givens with units → diagram → principle → algebra → substitution → answer and check. Use paper or add a text cell.  
*Full worked solution:* Module 12 P10 in the solutions collection (file `Week_12_Python_Solutions.ipynb`, opens 18 December 2026).


## Solutions / Çözümler

Complete worked solutions: Module 12 (Resonance), file `Week_12_Python_Solutions.ipynb`, opens 18 December 2026 on the course page.

[Course page / Ders sayfası](https://arifsolmaz.github.io/courses/fall/phy101/web/PHY101_Course_Dashboard.html)